# M1 · The data and the goal

**Tiny Transformer, Unit 1, episode M1.** What our tiny transformer learned from, and what its training makes small. Here you
load Tiny Shakespeare and count its characters, split it into the training text and the held-back text, cut a window into its
32 questions, score each of the model's answers on our line, average those scores over the whole held-back text, and compare
the model before and after training.

[Open in Google Colab](https://colab.research.google.com/github/gkiri/tiny_repo/blob/main/tiny-transformer/notebooks/M1-training-data.ipynb) (execution uses Google's hosted runtime; no graphics card needed). On your own computer:
`pip install -r requirements.txt`, then `jupyter notebook M1-training-data.ipynb`.

The notebook follows the page: two sections, six chapters, and in each chapter the page's three steps, *Picture it*, *Watch it
on our line* and *Read and run the code*, then a cell to run and a turn of your own.

**Computed here, and recorded.** Every number below is computed in this notebook, from the text and the model. The trained model's
9,417 numbers are the course's recorded run (`prompter-r1`), loaded from `prompter.json`; its untrained numbers are rebuilt here
from the run's seed. The asserts compare what you compute with what the course recorded, so the page and this notebook cannot
disagree.

In [1]:
#@title Setup: run me first (one click) { display-mode: "form" }
# In Google Colab this fetches the course's labs (notebooks, data and the trained model) and moves into this notebook's
# folder, so every file is where the cells below look for it. On your own computer, inside a copy of the labs, it does nothing.
import os, subprocess, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/tiny_repo"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/gkiri/tiny_repo", "/content/tiny_repo"], check=True)
    os.chdir("/content/tiny_repo/tiny-transformer/notebooks")
print("ready:", os.path.basename(os.getcwd()))

ready: notebooks


In [2]:
import base64, hashlib, json, math, os, urllib.request
import numpy as np
import torch

torch.set_num_threads(1)     # a model this small runs fastest on one CPU thread
print(torch.__version__)

2.10.0+cpu


**Before section 1, load the model.** The text (Tiny Shakespeare, pinned by its sha256), the model's code (`model.py`, copied in
whole from the course: the same file that trained it), its trained numbers, and the episode's five teaching functions (the
course's `code/training_data.py`, unchanged): `char_counts`, `split`, `window`, `loss_of_answer` and `average_loss`. Both files
keep their line numbers, as the page numbers them.

In [3]:
URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
SHA256 = "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed"
here = os.path.abspath('')
candidates = [os.path.join(here, '..', 'data', 'tinyshakespeare.txt'),                                   # in the labs
              os.path.join(here, '..', '..', '..', '..', 'training', 'prompter', 'data', 'tinyshakespeare.txt')]   # in the course
found = [p for p in candidates if os.path.exists(p)]
raw = open(found[0], 'rb').read() if found else urllib.request.urlopen(URL, timeout=60).read()
assert hashlib.sha256(raw).hexdigest() == SHA256, "not the pinned Tiny Shakespeare"
text = raw.decode('utf-8')

In [4]:
"""Our tiny transformer: 9,417 learned numbers that predict the next character of Shakespeare.

It reads up to 32 characters of text and gives a probability to each of the 65 characters that could come next. Writing is
that prediction in a loop: pick a character, append it, slide the 32-character window, and predict again. (The class is
called Prompter for the course's file names; learners meet it as "our tiny transformer".)

The architecture is the "10k" configuration of maxpolaczuk/tiny-character-transformer (a 1-block, 2-head character GPT on Tiny
Shakespeare). The code here is written fresh for the course (that repository has no licence, so none of its code or weights
are copied): every line is one the course teaches.

    characters ─► token table (65 × 28) + position table (32 × 28)        lookup:      what, and where
               ─► one block: LayerNorm ─► 2-head causal attention ─► +    look back:   which earlier letters matter
                             LayerNorm ─► feed-forward 28 → 56 → 28 ─► +  think:       transform each row on its own
               ─► final LayerNorm ─► readout with the token table again   score all 65 characters
               ─► softmax                                                  probabilities for the next character

Parameters: token table 1,820 · positions 896 · block 6,580 (two LayerNorms 112, q|k|v 2,436, attention out 812,
feed-forward 1,624 + 1,596) · final LayerNorm 56 · readout bias 65 = 9,417.
"""

import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass(frozen=True)
class Config:
    vocab_size: int = 65    # every distinct character in Tiny Shakespeare
    block_size: int = 32    # how many characters the model can read at once (its context)
    d_model: int = 28       # how many numbers describe one character in one place (a row)
    n_head: int = 2         # attention heads, each looking with 28 / 2 = 14 numbers
    d_ff: int = 56          # the feed-forward network's hidden width (2 × d_model)
    n_layer: int = 1        # transformer blocks

    def to_dict(self):
        return asdict(self)


class Tokenizer:
    """One id per character, in sorted order: '\\n' is 0, ' ' is 1, … 'z' is 64."""

    def __init__(self, text: str):
        self.chars = sorted(set(text))
        self.stoi = {c: i for i, c in enumerate(self.chars)}

    def encode(self, s: str) -> list[int]:
        return [self.stoi[c] for c in s]

    def decode(self, ids) -> str:
        return "".join(self.chars[int(i)] for i in ids)


class CausalSelfAttention(nn.Module):
    """Each row asks a question (q), every earlier row offers a label (k) and a message (v). A row reads the messages of the
    rows whose labels match its question best, never a row to its right."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.n_head, self.head_dim = cfg.n_head, cfg.d_model // cfg.n_head
        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model)     # q, k and v for both heads in one matrix
        self.proj = nn.Linear(cfg.d_model, cfg.d_model)        # mix the heads' results back into one row
        mask = torch.tril(torch.ones(cfg.block_size, cfg.block_size))
        self.register_buffer("mask", mask.view(1, 1, cfg.block_size, cfg.block_size), persistent=False)

    def forward(self, x, keep=None):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.n_head, self.head_dim).transpose(1, 2) for t in (q, k, v))
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)          # how well each question matches each label
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))   # no peeking at the future
        att = F.softmax(att, dim=-1)                                          # shares that add up to 1
        if keep is not None:
            keep["attn"] = att.detach()
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)            # each row: a weighted mix of messages
        return self.proj(y)


class Block(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ff = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_ff), nn.GELU(), nn.Linear(cfg.d_ff, cfg.d_model))

    def forward(self, x, keep=None):
        x = x + self.attn(self.ln1(x), keep)    # look back, and add what was found to the row
        x = x + self.ff(self.ln2(x))            # think about each row on its own, and add that too
        return x


class Prompter(nn.Module):
    def __init__(self, cfg: Config = Config()):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)    # one learned row per character
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.d_model)    # one learned row per place 0 … 31
        self.blocks = nn.Sequential(*[Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head_bias = nn.Parameter(torch.zeros(cfg.vocab_size))  # the readout reuses tok_emb (tied); only its bias is new
        self.apply(self._init)

    @staticmethod
    def _init(m):
        # small random numbers (σ = 0.02), so the untrained model's guesses are nearly even: its first loss is about
        # ln 65 = 4.17, the loss of a uniform guess over 65 characters
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, keep=None):
        T = idx.size(1)
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))   # what + where
        for b in self.blocks:
            x = b(x, keep)
        x = self.ln_f(x)
        logits = x @ self.tok_emb.weight.T + self.head_bias                         # a score for every character
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, n, temperature=1.0, generator=None):
        for _ in range(n):
            logits, _ = self(idx[:, -self.cfg.block_size:])
            probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
            nxt = torch.multinomial(probs, 1, generator=generator)
            idx = torch.cat([idx, nxt], dim=1)
        return idx


def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

In [5]:
tok = Tokenizer(text)                                   # the 65-character vocabulary (F0)
cfg = Config(vocab_size=len(tok.chars))
paths = [os.path.join(here, '..', 'model', 'prompter.json'),                                                     # in the labs
         os.path.join(here, '..', '..', '..', '..', 'src', 'specimen', 'tiny-transformer', 'data', 'prompter.json')]   # in the course
found = [p for p in paths if os.path.exists(p)]
assert found, "the model's trained numbers are not here: run the Setup cell at the top"
exp = json.load(open(found[0]))
state = {t['name']: torch.tensor(np.frombuffer(base64.b64decode(t['data']), dtype='<f4').reshape(t['shape']).copy())
         for t in exp['weights']['tensors']}
model = Prompter(cfg)
model.load_state_dict(state)
model.eval()
print(f"run {exp['run']}, step {exp['checkpoint_step']:,}: {count_params(model):,} learned numbers")
assert count_params(model) == 9417 and exp['checkpoint_step'] == 98000

run prompter-r1, step 98,000: 9,417 learned numbers


In [6]:
"""M1 · The data and the goal (Tiny Transformer, Unit 1; ADR 0011).

Five teaching functions say what our tiny transformer learns from and what training makes small, on top of
training/prompter/model.py: how often each of the 65 characters comes up; the 90/10 split of Tiny Shakespeare into training
text and held-back text; a window of 33 characters cut into the input x [32] and the targets y [32] (32 questions); the loss
of one answer, minus the natural log of the probability given to the right character; and the average loss over a whole text,
read in windows side by side, which is how the recorded run (prompter-r1) scored itself. The episode's notebook (notebooks/M1-training-data.ipynb) carries this file
unchanged. test_training_data.py checks it against the recorded run's own code (training/prompter/data.py split, train.py
windows and full_loss), against torch's cross-entropy, and against the recorded numbers (prompter-data.json,
prompter-training.json, prompter-traces.json).
"""
import torch


def char_counts(ids, V=65):
    """How often each of the V characters comes up: one count per id."""
    return torch.bincount(ids, minlength=V)                  # adds 1 to an id's count each time it appears      [65]


def split(ids, fraction=0.9):
    """The first 90% of the text is the training text; the last 10% is held back, never trained on."""
    n = int(fraction * len(ids))                             # where the held-back part starts: 1,003,854
    return ids[:n], ids[n:]                                  # [1003854], [111540]


def window(ids, start, T=32):
    """A window of T + 1 = 33 characters, cut into the input x and the targets y: the same window moved one along."""
    x = ids[start:start + T]                                 # the input: 32 characters                       [32]
    y = ids[start + 1:start + T + 1]                         # the targets: the next character at each place  [32]
    return x, y


def loss_of_answer(p):
    """The loss of an answer: minus the natural log of the probability p given to the right character."""
    return -torch.log(p)                                     # p = 1 → 0; p near 0 → large                    p's shape


@torch.no_grad()
def average_loss(model, ids, T=32, chunk=1024):
    """The average loss over a whole text, read in windows of T side by side: every character is scored once (but the last
    few, which do not fill a window), with the characters before it in its window."""
    starts = list(range(0, len(ids) - T - 1, T))             # window starts 0, 32, 64, …                     [W]
    total, count = 0.0, 0
    for a in range(0, len(starts), chunk):                   # a chunk of windows at a time, to save memory
        x, y = zip(*(window(ids, s, T) for s in starts[a:a + chunk]))
        x, y = torch.stack(x), torch.stack(y)                # a batch of windows                             [B, 32], [B, 32]
        logits, _ = model(x)                                 # a score for every character, at every place    [B, 32, 65]
        p = torch.softmax(logits, dim=-1)                    # probabilities                                  [B, 32, 65]
        p_right = p.gather(-1, y.unsqueeze(-1)).squeeze(-1)  # at each place, the probability of its target   [B, 32]
        total += loss_of_answer(p_right).sum().item()
        count += p_right.numel()
    return total / count                                     # one number: the average loss

# Section 1 · The text

What the model learned from: the text itself, the part held back to test it, and the questions hidden in it. Each chapter below
follows its page: Picture it, Watch it on our line, then Read and run the code.

## M1.1 · The text: Tiny Shakespeare

*What text did it learn from?*

**Predict first** (the page asks it too, before its clip): Tiny Shakespeare has 1,115,394 characters, every one from the same
65. Which one comes up most often: `e`, the space, or `t`?

### M1.1.1 · Picture it

The model gets no dictionary, no grammar and no rules: it learns only from examples. To learn what follows `th`, one line
(*they that think*) shows it 3 times; a page shows it about 40 times, and a pattern appears. More text gives more examples,
and better guesses. (The page's counts here are made up to show the idea.)

### M1.1.2 · Watch it on our line

*(The page's recorded clip, in words.)*

Everything the model knows came from one file: *Tiny Shakespeare*, Shakespeare's plays run together, 1,115,394 characters,
every one a token from the same 65. **Training** means changing the model's 9,417 learned numbers, step by step, so that
its predictions on this text get better. The text is uneven: in the training part the space is 15.3% of all characters and
`e` 8.5%; the dollar sign appears once.

### M1.1.3 · Read and run the code

Counting is one function in `code/training_data.py` (copied in above), `char_counts`: its one line, `torch.bincount`, adds 1 to
an id's count each time it appears, so it returns 65 counts, one per character. The first cell loads the text; the second
counts the training part and prints the most and the least common.

In [7]:
lines = text.count('\n')
print(f"{len(text):,} characters in {lines:,} lines")
print(text[:text.index('\n\n', 200)])                 # how it starts
assert len(text) == 1115394 and lines == 40_000

1,115,394 characters in 40,000 lines
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.


In [8]:
ids = torch.tensor(tok.encode(text))                    # the whole text as ids
train_ids, held_ids = split(ids)                        # M1.2: the part it trains on, and the part held back
freq = char_counts(train_ids)                           # how often each id appears in the training text   [65]
order = freq.argsort(descending=True).tolist()          # ids, the most common first
print('ids', tuple(ids.shape), '· counts', tuple(freq.shape))
bar = lambda n: '█' * max(1, round(40 * n / freq.max().item()))
for k, i in enumerate(order):
    if k < 8 or k >= 60:
        print(f"{tok.chars[i]!r:5} {freq[i].item():7,}  {100 * freq[i].item() / len(train_ids):4.1f}%  {bar(freq[i].item())}")
    elif k == 8:
        print('   … 52 more …')
assert [(tok.chars[i], freq[i].item()) for i in order[:5]] == [(' ', 153275), ('e', 85496), ('t', 60384), ('o', 59521), ('a', 49718)]
assert [(tok.chars[i], freq[i].item()) for i in order[-5:]] == [('Z', 161), ('X', 112), ('3', 27), ('&', 3), ('$', 1)]
assert round(100 * freq[tok.stoi[' ']].item() / len(train_ids), 1) == 15.3 and round(100 * freq[tok.stoi['e']].item() / len(train_ids), 1) == 8.5

ids (1115394,) · counts (65,)
' '   153,275  15.3%  ████████████████████████████████████████
'e'    85,496   8.5%  ██████████████████████
't'    60,384   6.0%  ████████████████
'o'    59,521   5.9%  ████████████████
'a'    49,718   5.0%  █████████████
'h'    46,390   4.6%  ████████████
's'    44,820   4.5%  ████████████
'r'    44,271   4.4%  ████████████
   … 52 more …
'Z'       161   0.0%  █
'X'       112   0.0%  █
'3'        27   0.0%  █
'&'         3   0.0%  █
'$'         1   0.0%  █


### ✋ Your turn

Which character is the third most common in the training text? Put it in `answer` (one character).

In [9]:
answer = None   # TODO: use order, the ids sorted from the most common

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == tok.chars[order[2]], 'order[2] is the id of the third most common; tok.chars turns an id back into its character'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [10]:
#@title Show a solution { display-mode: "form" }
answer = tok.chars[order[2]]
print(repr(answer), freq[order[2]].item(), "times")

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == tok.chars[order[2]], 'order[2] is the id of the third most common; tok.chars turns an id back into its character'
    print("✓ right:", answer)

't' 60384 times
✓ right: t


## M1.2 · The split: 90% to train, 10% held back

*Which text does it learn from, and which is held back?*

**Predict first** (the page asks it too): the run holds back 10% of Tiny Shakespeare and never trains on it. Which 10% makes a
fair test: every tenth character, spread through the text; a random 10% of its lines; or one unbroken stretch, the last 10%?

### M1.2.1 · Picture it

Picture a learner that only memorises: a table of every 5 characters it has seen, and the one that came next. Quizzed on text it
has seen, it does well: every run is in its table; on new text it has nothing to go on. So a score on seen text rewards memory,
and a fair test needs text kept out of learning.

### M1.2.2 · Watch it on our line

*(The page's recorded clip, in words.)*

Not all of the text is used for training. The first 90%, 1,003,854 characters, is the **training text**. The last 10%,
111,540 characters, is **held back**: the model never trains on it, so its score there is the honest test of what it learned.
Our running line from F0 sits in the held-back part, 1,100,473 characters in, and occurs nowhere in the training text: when the
model gave `r` the highest probability, it was not remembering.

### M1.2.3 · Read and run the code

The split is one function, `split`: its first line finds where the held-back part starts, 90% of the way in; its return line
cuts the ids there. (The cell above already ran it, to count the training text.) Run the cell: 1,003,854 characters to train
on, 111,540 held back, and our line 1,100,473 characters in, 0 times in the training text.

In [11]:
print(f"training text {len(train_ids):,} · held back {len(held_ids):,} · together {len(ids):,}")
LINE = 'You taught me language; and my p'
at = text.find(LINE)
print(f"our line starts at character {at:,}, {at / len(text):.1%} of the way in; the held-back part starts at {len(train_ids):,}")
print(text[at - 9:at + 42])                             # who says it, and how the play goes on
in_training = text[:len(train_ids)].count(LINE)
print('times it occurs in the training text:', in_training)
assert (len(train_ids), len(held_ids)) == (1003854, 111540) and len(train_ids) + len(held_ids) == len(text)
assert at == 1100473 and at >= len(train_ids) and in_training == 0

training text 1,003,854 · held back 111,540 · together 1,115,394
our line starts at character 1,100,473, 98.7% of the way in; the held-back part starts at 1,003,854
CALIBAN:
You taught me language; and my profit on't
times it occurs in the training text: 0


### ✋ Your turn

Is the line `"O Romeo, Romeo! wherefore art th"` in the training text or held back? Put `'train'` or `'held back'` in
`answer`.

In [12]:
answer = None   # TODO: 'train' or 'held back' (text.find gives where it starts)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == ('train' if text.find('O Romeo, Romeo! wherefore art th') < len(train_ids) else 'held back'), 'compare text.find(...) with len(train_ids): below it is training text'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [13]:
#@title Show a solution { display-mode: "form" }
where = text.find('O Romeo, Romeo! wherefore art th')
answer = 'train' if where < len(train_ids) else 'held back'
print(f"{where:,}: {answer}")

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == ('train' if text.find('O Romeo, Romeo! wherefore art th') < len(train_ids) else 'held back'), 'compare text.find(...) with len(train_ids): below it is training text'
    print("✓ right:", answer)

478,139: train
✓ right: train


## M1.3 · The questions: 32 in every window

*Where are the questions in plain text?*

**Predict first** (the page's opening asks it, and this chapter answers it): the training text has 1,003,854 characters, and no
answers are marked in it. How many questions of the kind *what comes next?* does it hold?

### M1.3.1 · Picture it

To learn, you need questions with their answers, like flashcards: a question on the front (*Romeo and Juli*), its answer on the
back (`e`). Writing cards like that by hand, for a whole book, would take years, and nobody wrote any. Plain text is all the
model gets, so its cards must be hidden in the text itself.

### M1.3.2 · Watch it on our line

*(The page's recorded clip, in words.)*

Cut a window of 33 characters, one more than the context. Its first 32 characters are the input, **x**. The same window moved
one along gives the **targets**, **y**: the right next character at every position. Every position asks one question, *what
comes next?*, so one window holds 32 questions, and a window can start at 1,003,821 places in the training text.

### M1.3.3 · Read and run the code

Cutting a window is one function, `window`: its first line takes 32 ids from where the window starts, the input x; its second
line takes the same 32 moved one along, the targets y. Run the cells: the targets are the inputs shifted one place, plus one new
character at the end. Then the places a window can start, and the notebook stacks 64 windows, as one training step does:
2,048 questions at once.

In [14]:
x, y = window(ids, at)                                  # the window that starts at our line
print('x', tuple(x.shape), repr(tok.decode(x)))
print('y', tuple(y.shape), repr(tok.decode(y)))
for t in (0, 1, 2, 30, 31):
    print(f"position {t:2d}: input {tok.chars[x[t]]!r} → target {tok.chars[y[t]]!r}")
assert tuple(x.shape) == tuple(y.shape) == (32,) and torch.equal(y[:-1], x[1:])   # y is x moved one along;
assert y[-1] == ids[at + 32]                                                       # its last target comes after x
assert tok.decode(x) == 'You taught me language; and my p' and tok.decode(y) == 'ou taught me language; and my pr'

x (32,) 'You taught me language; and my p'
y (32,) 'ou taught me language; and my pr'
position  0: input 'Y' → target 'o'
position  1: input 'o' → target 'u'
position  2: input 'u' → target ' '
position 30: input ' ' → target 'p'
position 31: input 'p' → target 'r'


Training cuts its windows at random places in the training text. The recorded run draws each start with
`torch.randint(len(train_ids) - 33, …)`, a whole number from 0 up to one below that, so this many places can start a window:

In [15]:
places = len(train_ids) - cfg.block_size - 1            # 0, 1, …, places − 1
print(f"{places:,} places to start a window; {cfg.block_size} questions in each")
starts = torch.randint(places, (64,), generator=torch.Generator().manual_seed(0)).tolist()   # one training step's starts
xb, yb = (torch.stack(t) for t in zip(*(window(train_ids, s) for s in starts)))
print('one training step:', tuple(xb.shape), tuple(yb.shape), '·', yb.numel(), 'questions')
assert places == 1003821 and tuple(xb.shape) == tuple(yb.shape) == (64, 32) and yb.numel() == 2048

1,003,821 places to start a window; 32 questions in each
one training step: (64, 32) (64, 32) · 2048 questions


### ✋ Your turn

Cut the window at the very start of the text, `window(ids, 0)`. What is the target at position 5? Put the character in
`answer`.

In [16]:
answer = None   # TODO: x0, y0 = window(ids, 0); then the character of y0[5]

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == tok.chars[window(ids, 0)[1][5]], 'the target at position 5 is y0[5]: the character right after x0[5]'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [17]:
#@title Show a solution { display-mode: "form" }
x0, y0 = window(ids, 0)
answer = tok.chars[y0[5]]
print(repr(tok.decode(x0[:6])), "→", repr(answer))

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == tok.chars[window(ids, 0)[1][5]], 'the target at position 5 is y0[5]: the character right after x0[5]'
    print("✓ right:", answer)

'First ' → 'C'
✓ right: C


# Section 2 · The score

How an answer is scored: the loss of one answer, the average that training makes small, and what training changed. Each chapter
again follows its page.

## M1.4 · The score: the loss of one answer

*How is one answer scored?*

**Predict first** (the page asks it too): after *“You ta”*, the model gave the right next character, `u`, only 0.004. What does
that answer cost: about 2.8 (just past the ladder's last step), about 5.5 (about 8 halvings from 1), or about 250 (1 divided by
0.004)?

### M1.4.1 · Picture it

The model never says just right or wrong: it gives the right answer a probability, from 0 to 1. Minus the natural log turns it
into a cost: 1 costs 0, a half 0.69, a quarter 1.39, an eighth 2.08. Each halving adds the same 0.69, so a tiny probability
costs a lot.

### M1.4.2 · Watch it on our line

*(The page's recorded clip, in words.)*

At each position the model gives the right next character some probability $p$. The **loss** of that answer is minus its natural
log: $\text{loss} = -\ln p$. A sure, right answer ($p$ near 1) scores near 0; a confident wrong one ($p$ near 0) scores high.
Lower is better. One forward pass answers all 32 questions of the window at once: the logits are `[32×65]`, a row per position
(`[1×32×65]` here, for a batch of one window).

### M1.4.3 · Read and run the code

The loss of an answer is one function, `loss_of_answer`: its one line returns minus the natural log of `p`. Run the cell on our
line's 32 answers: the losses run from 0.05 to 5.45. The cell checks them against torch's own
`cross_entropy`, the loss training averages: the same 32 numbers.

In [18]:
with torch.no_grad():
    logits, _ = model(x.unsqueeze(0))                   # the window as a batch of one               [1, 32, 65]
probs = torch.softmax(logits[0], dim=-1)                # 65 probabilities at every position         [32, 65]
p_right = probs[torch.arange(32), y]                    # each position's probability of its target  [32]
losses = loss_of_answer(p_right)                        # [32]
print('logits', tuple(logits.shape), '· p_right', tuple(p_right.shape), '· losses', tuple(losses.shape))
for t in range(32):
    print(f"{t:2d}  after {LINE[max(0, t - 7):t + 1]!r:11} target {tok.chars[y[t]]!r:4}  p {p_right[t].item():.4f}  loss {losses[t].item():.3f}")
RECORDED = [0.1733, 0.7093, 0.2455, 0.0983, 0.017, 0.0043, 0.2172, 0.9314, 0.75, 0.5331, 0.0657, 0.2263, 0.4414, 0.0316, 0.1941, 0.1386, 0.0315, 0.0822, 0.1247, 0.0167, 0.5327, 0.0362, 0.2521, 0.1886, 0.6263, 0.9139, 0.9479, 0.0792, 0.2682, 0.9127, 0.0263, 0.3403]
assert tuple(logits.shape) == (1, 32, 65) and tuple(p_right.shape) == (32,)
assert all(abs(a - b) < 2e-4 for a, b in zip(p_right.tolist(), RECORDED))
assert p_right[7] > 0.9 and p_right[5] < 0.01           # after "You taug", h is near sure; after "You ta", u is a surprise
assert [round(losses[t].item(), 2) for t in (7, 5, 31)] == [round(-math.log(RECORDED[t]), 2) for t in (7, 5, 31)]
ce = torch.nn.functional.cross_entropy(logits[0], y, reduction='none')   # torch's own loss, one per question   [32]
print(f"losses {losses.min().item():.2f} … {losses.max().item():.2f} · torch's cross_entropy the same:", torch.allclose(ce, losses, atol=1e-5))
assert torch.allclose(ce, losses, atol=1e-5) and (round(losses.min().item(), 2), round(losses.max().item(), 2)) == (0.05, 5.45)

logits (1, 32, 65) · p_right (32,) · losses (32,)
 0  after 'Y'         target 'o'   p 0.1733  loss 1.753
 1  after 'Yo'        target 'u'   p 0.7093  loss 0.343
 2  after 'You'       target ' '   p 0.2455  loss 1.404
 3  after 'You '      target 't'   p 0.0983  loss 2.320
 4  after 'You t'     target 'a'   p 0.0170  loss 4.075
 5  after 'You ta'    target 'u'   p 0.0043  loss 5.450
 6  after 'You tau'   target 'g'   p 0.2172  loss 1.527
 7  after 'You taug'  target 'h'   p 0.9314  loss 0.071
 8  after 'ou taugh'  target 't'   p 0.7500  loss 0.288
 9  after 'u taught'  target ' '   p 0.5331  loss 0.629
10  after ' taught '  target 'm'   p 0.0657  loss 2.723
11  after 'taught m'  target 'e'   p 0.2263  loss 1.486
12  after 'aught me'  target ' '   p 0.4414  loss 0.818
13  after 'ught me '  target 'l'   p 0.0316  loss 3.453
14  after 'ght me l'  target 'a'   p 0.1941  loss 1.639
15  after 'ht me la'  target 'n'   p 0.1386  loss 1.976
16  after 't me lan'  target 'g'   p 0.0315  loss 3.45

### ✋ Your turn

Compute a loss by hand. At position 28 the input is `m` (of *my*) and the target is `y`. Use `math.log` and
`p_right[28].item()` to compute its loss, and put it, rounded to 2 places, in `answer`.

In [19]:
answer = None   # TODO: round(-math.log(...), 2)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == round(-math.log(p_right[28].item()), 2), 'the loss is minus the natural log: -math.log(p), then round(…, 2)'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [20]:
#@title Show a solution { display-mode: "form" }
answer = round(-math.log(p_right[28].item()), 2)
print(f"p {p_right[28].item():.4f} → loss {answer}")

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == round(-math.log(p_right[28].item()), 2), 'the loss is minus the natural log: -math.log(p), then round(…, 2)'
    print("✓ right:", answer)

p 0.2682 → loss 1.32
✓ right: 1.32


## M1.5 · The goal: the average loss on held-back text

*What number does training make small?*

**Predict first** (the page asks it too): on held-back text, an even guess scores 4.17, and counting which character follows
which (counting pairs) 2.48. Where does the trained model land: below counting pairs (about 1.9), between the two (about 3.3), or near 0?

### M1.5.1 · Picture it

One answer has one cost; a whole text needs one number, the average. Is an average good or bad? A number alone cannot say:
compare it with plain guessing. An even guess among 4 characters gives each a quarter and pays $\ln 4 \approx 1.39$ on
every answer. Below the even guess, something was learned; only a sure, right answer every time scores 0.

### M1.5.2 · Watch it on our line

*(The page's recorded clip, in words.)*

Average the losses over many answers and you get one number. Training makes it small on the training text; the same average on
the held-back text says whether what the model learned carries over. On the held-back text an even guess scores
$\ln 65 \approx 4.17$, counting pairs 2.48, and the trained model 1.93: far from perfect, but far better than guessing, on text it
never saw.

### M1.5.3 · Read and run the code

Scoring a whole text is one function, `average_loss`: it reads the text in windows of 32 side by side, runs the model on a chunk
at a time, adds up every answer's loss, and returns the average. The cells below run it on the held-back text and on the
training text, and compute the two yardsticks, the even guess and counting pairs, in full.

### The three scores, in full

Three scores on the held-back text, each computed below in full:

- **An even guess** gives every character $1/65$, so every answer's loss is $\ln 65$.
- **Counting pairs** counts which character follows which in the training text, adds 1 to every count, and turns each row into
  probabilities. Then it scores each neighbouring pair of the held-back text. (F1 builds this counter by hand.)
- **The trained model** reads the held-back text in windows of 32 side by side, and every answer in every window is scored. This
  is how the recorded run scored itself at the end.

$e^{-\text{loss}}$ turns an average loss back into a probability: the typical probability given to the right answer (the
geometric mean of all of them).

In [21]:
V = len(tok.chars)
even = math.log(V)                                      # every answer: −ln(1/65) = ln 65
counts = torch.zeros(V, V)                              # counts[a, b]: how often b came right after a     [65, 65]
counts.index_put_((train_ids[:-1], train_ids[1:]), torch.ones(len(train_ids) - 1), accumulate=True)   # add 1 per pair
pair_p = (counts + 1) / (counts + 1).sum(dim=1, keepdim=True)   # add 1 to every count; each row sums to 1   [65, 65]
pair_loss = loss_of_answer(pair_p[held_ids[:-1], held_ids[1:]]).mean().item()   # every held-back pair
model_loss = average_loss(model, held_ids)              # the trained model, windows side by side
train_loss = average_loss(model, train_ids)             # the same, on the training text
windows = len(range(0, len(held_ids) - cfg.block_size - 1, cfg.block_size))
print(f"held back: {len(held_ids) - 1:,} pairs; {windows:,} windows of 32 = {windows * 32:,} answers")
for name, loss in (('an even guess', even), ('counting pairs', pair_loss), ('the trained model', model_loss)):
    print(f"{name:18} loss {loss:.4f}   typical probability e^-loss {math.exp(-loss):.3f}")
print(f"(the trained model on its own training text: {train_loss:.4f})")

held back: 111,539 pairs; 3,485 windows of 32 = 111,520 answers
an even guess      loss 4.1744   typical probability e^-loss 0.015
counting pairs     loss 2.4819   typical probability e^-loss 0.084
the trained model  loss 1.9317   typical probability e^-loss 0.145
(the trained model on its own training text: 1.7651)


In [22]:
RECORDED = {'uniform_loss': 4.1744, 'bigram_val_loss': 2.4819, 'val_loss': 1.9317, 'train_loss': 1.7651}
assert abs(even - RECORDED['uniform_loss']) < 1e-4 and abs(math.exp(-even) - 1 / 65) < 1e-12
assert abs(pair_loss - RECORDED['bigram_val_loss']) < 2e-4
assert abs(model_loss - RECORDED['val_loss']) < 2e-4 and abs(train_loss - RECORDED['train_loss']) < 2e-4
assert model_loss < pair_loss < even and windows * 32 == 111520
print([round(v, 2) for v in (even, pair_loss, model_loss)])

[4.17, 2.48, 1.93]


### ✋ Your turn

The typical probability of a right answer for counting pairs: compute $e^{-\text{loss}}$ from `pair_loss` with
`math.exp`, and put it, rounded to 3 places, in `answer`. How many times the even guess's $1/65$ is it?

In [23]:
answer = None   # TODO: round(math.exp(-pair_loss), 3)

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == round(math.exp(-pair_loss), 3), 'math.exp(-pair_loss), then round(…, 3)'
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [24]:
#@title Show a solution { display-mode: "form" }
answer = round(math.exp(-pair_loss), 3)
print(answer, "·", round(math.exp(-pair_loss) * 65, 1), "times an even guess")

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert answer == round(math.exp(-pair_loss), 3), 'math.exp(-pair_loss), then round(…, 3)'
    print("✓ right:", answer)

0.084 · 5.4 times an even guess
✓ right: 0.084


## M1.6 · Before and after: only the numbers changed

*What does training change?*

**Predict first** (the page asks it too): before training, the model's 9,417 learned numbers are small and random. What is
its loss on held-back text: about 4.2 (no better than an even guess), about 2.5, or far above 10?

### M1.6.1 · Picture it

Picture the model as a machine: fixed wiring, and many knobs. The wiring is the code; the knobs are the learned numbers. Keep the
wiring and turn the knobs, and a different answer comes out. Training turns only the knobs, never the wiring; our model has
9,417 of them.

### M1.6.2 · Watch it on our line

*(The page's recorded clip, in words.)*

Training changes the 9,417 learned numbers, and nothing else. The same code twice: untrained, its numbers as they started,
small and random, it writes letter soup, and its loss is 4.18, about the even guess. Trained, after 98,000 steps, the
same code writes words, speakers' names in capitals, and line breaks. The code did not change; only the numbers did.

### M1.6.3 · Read and run the code

The starting numbers are set in `model.py`, by its `_init`: most weights (learned numbers that multiply) start small and
random, spread 0.02 around 0, and every bias (one that is added) starts at 0. The cell sets the run's seed, 1337 (the number
that makes random draws repeatable), gets back the run's own numbers at step 0, and counts what training changed: all
9,417 of them.

### The same code, twice

Training changes the 9,417 learned numbers and nothing else. Below is the same code twice. *Before*: `torch.manual_seed(1337)`,
then build the model, exactly as the recorded run started, so these are its numbers at step 0, small and random. *After*: the
trained numbers, at step 98,000. Each writes 240 characters after a new line, with the same seed, 42.

In [25]:
torch.manual_seed(1337)                                 # the recorded run's seed, set before the model is built
untrained = Prompter(cfg).eval()                        # step 0: small random numbers
STEP0 = {'head_bias': 0.0, 'tok_emb.weight': 28.3645, 'pos_emb.weight': 13.7808, 'blocks.0.ln1.weight': 28.0, 'blocks.0.ln1.bias': 0.0, 'blocks.0.attn.qkv.weight': 36.7723, 'blocks.0.attn.qkv.bias': 0.0, 'blocks.0.attn.proj.weight': 12.8984, 'blocks.0.attn.proj.bias': 0.0, 'blocks.0.ln2.weight': 28.0, 'blocks.0.ln2.bias': 0.0, 'blocks.0.ff.0.weight': 25.1978, 'blocks.0.ff.0.bias': 0.0, 'blocks.0.ff.2.weight': 24.7042, 'blocks.0.ff.2.bias': 0.0, 'ln_f.weight': 28.0, 'ln_f.bias': 0.0}
same_start = all(abs(v.double().abs().sum().item() - STEP0[k]) < 1e-3 for k, v in untrained.state_dict().items())
print("the recorded run's step-0 numbers:", same_start)
changed = sum((a != b).sum().item() for a, b in zip(untrained.parameters(), model.parameters()))
print(f"{changed:,} of {count_params(model):,} numbers differ after training; the code is the same:", type(untrained) is type(model))
untrained_loss = average_loss(untrained, held_ids)
print(f"held-back loss before {untrained_loss:.2f} (an even guess {even:.2f}) · after {model_loss:.2f}")
assert same_start and list(STEP0) == list(untrained.state_dict()) and changed == 9417
assert abs(untrained_loss - 4.178) < 0.005        # the page's step-0 loss, 4.178 (the run's own evaluation batches)

the recorded run's step-0 numbers: True
9,417 of 9,417 numbers differ after training; the code is the same: True


held-back loss before 4.18 (an even guess 4.17) · after 1.93


In [26]:
start = torch.tensor([tok.encode('\n')])               # a new line
before = tok.decode(untrained.generate(start, 240, generator=torch.Generator().manual_seed(42))[0, 1:])
after = tok.decode(model.generate(start, 240, generator=torch.Generator().manual_seed(42))[0, 1:])
print('--- before training (step 0)')
print(before)
print('--- after training (step 98,000)')
print(after)
assert before == "v:hDYqLMfJ\nvdfCaN:$wP &IAW:cGdn.,pSzoWDnqCOnmc.G HVyye&zaXMENFd\nlvmhgUX!qdrdRWI$k-\ntC;oDG;uxgRl3AB?i\nuwF!SWTqQ'u\nceWAiA3 \nKmUIQj!:U&vaD'&$RLUzJhDctR!MM?bNIgk-rfng O?Wrz\nSFPvnHWXSGQc,u-bQLU.U&hOVz:,?bqcQW?E$cUdHPR ;tHuTXFOoHN;CNZWCg;mWlK,uFL"
assert after == '\nAs loud save may: whom grows no, onoth, but court of in and go lamstre!\n\nKISS:\nWhat all thegnings?\n\nDUKE VINCENTIO:\nAway\nKING HE:\nIt hand the how by the your for not prenatnge, likes both.\n\nPOMEOK:\nThat?\n\nGLOUCETES:\nThe of his he like ward'

--- before training (step 0)
v:hDYqLMfJ
vdfCaN:$wP &IAW:cGdn.,pSzoWDnqCOnmc.G HVyye&zaXMENFd
lvmhgUX!qdrdRWI$k-
tC;oDG;uxgRl3AB?i
uwF!SWTqQ'u
ceWAiA3 
KmUIQj!:U&vaD'&$RLUzJhDctR!MM?bNIgk-rfng O?Wrz
SFPvnHWXSGQc,u-bQLU.U&hOVz:,?bqcQW?E$cUdHPR ;tHuTXFOoHN;CNZWCg;mWlK,uFL
--- after training (step 98,000)

As loud save may: whom grows no, onoth, but court of in and go lamstre!

KISS:
What all thegnings?

DUKE VINCENTIO:
Away
KING HE:
It hand the how by the your for not prenatnge, likes both.

POMEOK:
That?

GLOUCETES:
The of his he like ward


### ✋ Your turn

Score the untrained model on our line: with `untrained`, `x`, `y` and `loss_of_answer`, compute the average loss of its 32
answers and put it, rounded to 2 places, in `answer`. Is it near $\ln 65$?

In [27]:
answer = None   # TODO: run untrained on x, take each target's probability, average the losses

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - loss_of_answer(torch.softmax(untrained(x.unsqueeze(0))[0][0], dim=-1)[torch.arange(32), y]).mean().item()) < 0.006, "as in section 4, but with untrained: softmax of its logits, the targets' probabilities, loss_of_answer, .mean()"
    print("✓ right:", answer)

Your turn: fill in `answer` above, then run this cell again.


In [28]:
#@title Show a solution { display-mode: "form" }
with torch.no_grad():
    p_untrained = torch.softmax(untrained(x.unsqueeze(0))[0][0], dim=-1)[torch.arange(32), y]
answer = round(loss_of_answer(p_untrained).mean().item(), 2)
print(answer, "· ln 65 =", round(math.log(65), 2))

if answer is None:
    print("Your turn: fill in `answer` above, then run this cell again.")
else:
    assert abs(answer - loss_of_answer(torch.softmax(untrained(x.unsqueeze(0))[0][0], dim=-1)[torch.arange(32), y]).mean().item()) < 0.006, "as in section 4, but with untrained: softmax of its logits, the targets' probabilities, loss_of_answer, .mean()"
    print("✓ right:", answer)

4.22 · ln 65 = 4.17
✓ right: 4.22


**Next, M2:** open the box, and follow our line through it, starting with the embeddings.